# ADS2001 - Data challenges 3 - S1 2026

## Melbourne Climate Data

### By Winston Wijaya

### Group Members: Jiehni Koh, Zewen Zhao, Jiajie Zhu

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [6]:
# For reading the Excel File
print("="*60)
print("MELBOURNE WEATHER DATA ANALYSIS")
print("="*60)

MELBOURNE WEATHER DATA ANALYSIS


In [7]:
# Loading the data
df = pd.read_excel(r"C:\Users\winst\OneDrive\Documents\University\Year 2\Semester 1\ADS2001\predicting_melbourne_weather\Melbourne01.xlsx")
print(f"\n✓ Data loaded successfully!")
print(f"  Dataset shape: {df.shape}")


✓ Data loaded successfully!
  Dataset shape: (505355, 18)


In [8]:
# Display first few rows
print("\n" + "="*60)
print("FIRST 5 ROWS OF RAW DATA")
print("="*60)
print(df.head())

# Display column names
print("\n" + "="*60)
print("COLUMN NAMES")
print("="*60)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

# Remove missing values
df = df.dropna()
print(f"\n✓ After dropping missing values: {df.shape}")



FIRST 5 ROWS OF RAW DATA
   Year Month  Day  Hour   Min  Air Temp (degrees C)  \
0  2011     1  1.0   0.0   4.0                  24.8   
1  2011     1  1.0   0.0  14.0                  24.8   
2  2011     1  1.0   0.0  24.0                  24.9   
3  2011     1  1.0   0.0  34.0                  24.7   
4  2011     1  1.0   0.0  44.0                  24.1   

   Apparent Temp (degrees C)  Dew Pt Temp (degrees C)  Humidity (%)  \
0                        0.0                     14.0          51.0   
1                        0.0                     13.3          48.0   
2                        0.0                     13.3          48.0   
3                        0.0                     13.4          49.0   
4                        0.0                     13.3          51.0   

  Wind Direction Wind Speed (km/h)  Wind Gust  (km/h)  MSLP (hPa)  \
0             SE                11               13.0      1007.4   
1             SE                11               11.0      1007.5   
2  

In [9]:
# Create a copy for processing
data = df.copy()

In [43]:
# Convert date/time columns to integers
print("\n" + "="*60)
print("CONVERTING DATE/TIME COLUMNS")
print("="*60)

data['Year'] = data['Year'].astype(int)
data['Month'] = data['Month'].astype(int)
data['Day'] = data['Day'].astype(int)
data['Hour'] = data['Hour'].fillna(0).astype(int)
data['Min'] = data['Min'].fillna(0).astype(int)

print("✓ Date/time columns converted to integers")


CONVERTING DATE/TIME COLUMNS
✓ Date/time columns converted to integers


In [21]:
# Create datetime column using string concatenation
print("\n" + "="*60)
print("CREATING DATETIME COLUMN")
print("="*60)

# Method: Combine components into a string and convert to datetime
data['Datetime'] = pd.to_datetime(
    data['Year'].astype(str) + '-' + 
    data['Month'].astype(str).str.zfill(2) + '-' + 
    data['Day'].astype(str).str.zfill(2) + ' ' + 
    data['Hour'].astype(str).str.zfill(2) + ':' + 
    data['Min'].astype(str).str.zfill(2),
    format='%Y-%m-%d %H:%M',
    errors='coerce'
)

# Check for any invalid datetimes
if data['Datetime'].isnull().any():
    print(f"⚠️ Warning: {data['Datetime'].isnull().sum()} invalid datetime entries found and will be dropped")
    data = data.dropna(subset=['Datetime'])

# Extract date only (for daily grouping)
data['Date_only'] = data['Datetime'].dt.date

print(f"✓ Datetime created successfully")
print(f"  Date range: {data['Datetime'].min()} to {data['Datetime'].max()}")
print(f"  Number of unique dates: {data['Date_only'].nunique()}")
print(f"  Total rows after datetime validation: {len(data)}")


CREATING DATETIME COLUMN
✓ Datetime created successfully
  Date range: 2011-01-01 00:04:00 to 2022-03-08 16:20:00
  Number of unique dates: 4078
  Total rows after datetime validation: 504500


In [22]:
# Convert problematic columns to numeric BEFORE checking numeric columns
print("\n" + "="*60)
print("CONVERTING ALL WEATHER COLUMNS TO NUMERIC")
print("="*60)

# List of all weather columns that should be numeric
weather_columns = [
    'Air Temp (degrees C)', 'Apparent Temp (degrees C)', 'Dew Pt Temp (degrees C)',
    'Humidity (%)', 'Wind Speed (km/h)', 'Wind Gust (km/h)', 'MSLP (hPa)',
    'Rainfall since 9 am (mm)', 'gamma', 'Calculated Dew Pt Temp (degrees C)',
    'E (hPa)', 'Calculated Apparent Temp (degrees C)', 'Rain'
]

# Convert each to numeric, coercing errors to NaN
for col in weather_columns:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
        print(f"  ✓ Converted {col} to numeric")

# Drop any rows with NaN in important columns (optional - can be adjusted)
initial_rows = len(data)
data = data.dropna(subset=['Air Temp (degrees C)', 'Rainfall since 9 am (mm)'])
print(f"\n✓ Dropped {initial_rows - len(data)} rows with missing critical data")
print(f"  Remaining rows: {len(data)}")


CONVERTING ALL WEATHER COLUMNS TO NUMERIC
  ✓ Converted Air Temp (degrees C) to numeric
  ✓ Converted Apparent Temp (degrees C) to numeric
  ✓ Converted Dew Pt Temp (degrees C) to numeric
  ✓ Converted Humidity (%) to numeric
  ✓ Converted Wind Speed (km/h) to numeric
  ✓ Converted MSLP (hPa) to numeric
  ✓ Converted Rainfall since 9 am (mm) to numeric
  ✓ Converted gamma to numeric
  ✓ Converted Calculated Dew Pt Temp (degrees C) to numeric
  ✓ Converted E (hPa) to numeric
  ✓ Converted Calculated Apparent Temp (degrees C) to numeric

✓ Dropped 0 rows with missing critical data
  Remaining rows: 504500


In [23]:
# Select only numeric columns for aggregation (exclude string columns)
print("\n" + "="*60)
print("SELECTING NUMERIC COLUMNS FOR AGGREGATION")
print("="*60)

# Identify numeric columns for weather data (exclude date/time columns)
columns_to_exclude = ['Year', 'Month', 'Day', 'Hour', 'Min', 'Datetime', 'Date_only', 
                      'Wind Direction', 'day_of_week', 'wind_sin', 'wind_cos', 'day_of_year']

numeric_columns = []
for col in data.columns:
    if col not in columns_to_exclude:
        if pd.api.types.is_numeric_dtype(data[col]):
            numeric_columns.append(col)
            print(f"  ✓ {col} - numeric")
        else:
            print(f"  ✗ {col} - non-numeric (skipping)")

print(f"\n✓ Found {len(numeric_columns)} numeric columns for aggregation")


SELECTING NUMERIC COLUMNS FOR AGGREGATION
  ✓ Air Temp (degrees C) - numeric
  ✓ Apparent Temp (degrees C) - numeric
  ✓ Dew Pt Temp (degrees C) - numeric
  ✓ Humidity (%) - numeric
  ✓ Wind Speed (km/h) - numeric
  ✓ Wind Gust  (km/h) - numeric
  ✓ MSLP (hPa) - numeric
  ✓ Rainfall since 9 am (mm) - numeric
  ✓ gamma - numeric
  ✓ Calculated Dew Pt Temp (degrees C) - numeric
  ✓ E (hPa) - numeric
  ✓ Calculated Apparent Temp (degrees C) - numeric

✓ Found 12 numeric columns for aggregation


In [24]:
# Create daily statistics with only numeric columns
print("\n" + "="*60)
print("CREATING DAILY STATISTICS")
print("="*60)

# Define aggregation functions for each numeric variable
daily_agg = {}
for col in numeric_columns:
    if col == 'Rainfall since 9 am (mm)' or col == 'Rain':
        daily_agg[col] = ['sum', 'max', 'count']
    elif col in ['Air Temp (degrees C)', 'Apparent Temp (degrees C)', 'MSLP (hPa)']:
        daily_agg[col] = ['mean', 'min', 'max']
    elif col in ['Humidity (%)', 'Wind Speed (km/h)']:
        daily_agg[col] = ['mean', 'max']
    elif col in ['Wind Gust (km/h)']:
        daily_agg[col] = ['max']
    else:
        daily_agg[col] = ['mean']

# Group by date and aggregate
daily_stats = data.groupby('Date_only').agg(daily_agg).round(2)

# Flatten column names
daily_stats.columns = ['_'.join(col).strip() for col in daily_stats.columns.values]

# Rename columns to be more readable
column_renames = {}
for col in daily_stats.columns:
    if 'Air Temp (degrees C)_mean' in col:
        column_renames[col] = 'temp_mean_c'
    elif 'Air Temp (degrees C)_min' in col:
        column_renames[col] = 'temp_min_c'
    elif 'Air Temp (degrees C)_max' in col:
        column_renames[col] = 'temp_max_c'
    elif 'Dew Pt Temp (degrees C)_mean' in col:
        column_renames[col] = 'dew_point_mean_c'
    elif 'Apparent Temp (degrees C)_mean' in col:
        column_renames[col] = 'apparent_temp_mean_c'
    elif 'Apparent Temp (degrees C)_min' in col:
        column_renames[col] = 'apparent_temp_min_c'
    elif 'Apparent Temp (degrees C)_max' in col:
        column_renames[col] = 'apparent_temp_max_c'
    elif 'Humidity (%)_mean' in col:
        column_renames[col] = 'humidity_mean_pct'
    elif 'Humidity (%)_max' in col:
        column_renames[col] = 'humidity_max_pct'
    elif 'Wind Speed (km/h)_mean' in col:
        column_renames[col] = 'wind_speed_mean_kmh'
    elif 'Wind Speed (km/h)_max' in col:
        column_renames[col] = 'wind_speed_max_kmh'
    elif 'Wind Gust (km/h)_max' in col:
        column_renames[col] = 'wind_gust_max_kmh'
    elif 'MSLP (hPa)_mean' in col:
        column_renames[col] = 'pressure_mean_hpa'
    elif 'MSLP (hPa)_min' in col:
        column_renames[col] = 'pressure_min_hpa'
    elif 'MSLP (hPa)_max' in col:
        column_renames[col] = 'pressure_max_hpa'
    elif 'Rainfall since 9 am (mm)_sum' in col:
        column_renames[col] = 'rainfall_daily_mm'
    elif 'Rainfall since 9 am (mm)_max' in col:
        column_renames[col] = 'rainfall_max_intensity_mm'
    elif 'Rainfall since 9 am (mm)_count' in col:
        column_renames[col] = 'rainfall_observations_count'
    elif 'Rain_sum' in col:
        column_renames[col] = 'rain_binary_sum'
    elif 'gamma_mean' in col:
        column_renames[col] = 'gamma_mean'
    elif 'Calculated Dew Pt Temp (degrees C)_mean' in col:
        column_renames[col] = 'calculated_dew_point_mean_c'
    elif 'E (hPa)_mean' in col:
        column_renames[col] = 'vapor_pressure_mean_hpa'
    elif 'Calculated Apparent Temp (degrees C)_mean' in col:
        column_renames[col] = 'calculated_apparent_temp_mean_c'

daily_stats = daily_stats.rename(columns=column_renames)

# Add derived columns - check if rainfall column exists
if 'rainfall_daily_mm' in daily_stats.columns:
    daily_stats['rain_occurred'] = (daily_stats['rainfall_daily_mm'] > 0).astype(int)
    print("✓ Added rain_occurred column")
elif 'rain_binary_sum' in daily_stats.columns:
    # Use the Rain binary column if available
    daily_stats['rain_occurred'] = (daily_stats['rain_binary_sum'] > 0).astype(int)
    print("✓ Added rain_occurred column from Rain data")
else:
    print("⚠️ Warning: No rainfall data found for rain_occurred column")

# Check if temperature columns exist before adding temperature range
if 'temp_max_c' in daily_stats.columns and 'temp_min_c' in daily_stats.columns:
    daily_stats['temperature_range_c'] = daily_stats['temp_max_c'] - daily_stats['temp_min_c']
    print("✓ Added temperature_range_c column")

# Check if pressure columns exist before adding pressure range
if 'pressure_max_hpa' in daily_stats.columns and 'pressure_min_hpa' in daily_stats.columns:
    daily_stats['pressure_range_hpa'] = daily_stats['pressure_max_hpa'] - daily_stats['pressure_min_hpa']
    print("✓ Added pressure_range_hpa column")

# Reset index to make Date_only a column
daily_stats = daily_stats.reset_index()
daily_stats['Date_only'] = pd.to_datetime(daily_stats['Date_only'])

print(f"\n✓ Daily statistics created successfully!")
print(f"  Shape: {daily_stats.shape}")
print(f"  Date range: {daily_stats['Date_only'].min().date()} to {daily_stats['Date_only'].max().date()}")
print(f"  Total days: {len(daily_stats)}")


CREATING DAILY STATISTICS
✓ Added rain_occurred column
✓ Added temperature_range_c column
✓ Added pressure_range_hpa column

✓ Daily statistics created successfully!
  Shape: (4078, 26)
  Date range: 2011-01-01 to 2022-03-08
  Total days: 4078


In [25]:
# Display first few rows of daily statistics
print("\n" + "="*60)
print("FIRST 10 ROWS OF DAILY STATISTICS")
print("="*60)
print(daily_stats.head(10))


FIRST 10 ROWS OF DAILY STATISTICS
   Date_only  temp_mean_c  temp_min_c  temp_max_c  apparent_temp_mean_c  \
0 2011-01-01        19.65        16.8        24.9                   0.0   
1 2011-01-02        18.17        15.3        21.4                   0.0   
2 2011-01-03        16.58        14.1        20.3                   0.0   
3 2011-01-04        16.78        12.9        21.5                   0.0   
4 2011-01-05        18.11        14.6        21.3                   0.0   
5 2011-01-06        22.26        15.9        30.6                   0.0   
6 2011-01-07        27.42        21.5        33.3                   0.0   
7 2011-01-08        26.62        19.4        34.3                   0.0   
8 2011-01-09        20.29        18.2        22.5                   0.0   
9 2011-01-10        21.50        19.8        24.9                   0.0   

   apparent_temp_min_c  apparent_temp_max_c  dew_point_mean_c  \
0                  0.0                  0.0             12.00   
1        

In [75]:
#Note: (WIP - Attemtping to fix the apparent_temp_c section with data gathered, because it's somehow all zeroes...)

print("\n" + "="*60)
print("CORRECTING DATA ISSUES")
print("="*60)

# 1. Fix Rainfall: We need MAX (not SUM) since it's cumulative since 9am
print("\nRecalculating rainfall correctly...")

# Make sure Date_only is consistent in both dataframes
data['Date_only'] = pd.to_datetime(data['Date_only'])

# Calculate correct daily rainfall (max value of the cumulative rainfall)
rainfall_corrected = data.groupby('Date_only')['Rainfall since 9 am (mm)'].max().round(2)
rainfall_corrected = rainfall_corrected.reset_index()

# Ensure both dataframes have the same data type for Date_only
daily_stats['Date_only'] = pd.to_datetime(daily_stats['Date_only'])
rainfall_corrected['Date_only'] = pd.to_datetime(rainfall_corrected['Date_only'])

# Rename the column
rainfall_corrected.columns = ['Date_only', 'rainfall_daily_mm_corrected']

# Merge corrected rainfall back
daily_stats = daily_stats.merge(rainfall_corrected, on='Date_only', how='left')

# Replace the incorrect rainfall values
daily_stats['rainfall_daily_mm'] = daily_stats['rainfall_daily_mm_corrected']
daily_stats = daily_stats.drop(columns=['rainfall_daily_mm_corrected'])

# Update rain_occurred
daily_stats['rain_occurred'] = (daily_stats['rainfall_daily_mm'] > 0).astype(int)

print("✓ Rainfall corrected: Now using daily maximum (not sum)")
print(f"  Total rainfall after correction: {daily_stats['rainfall_daily_mm'].sum():.1f} mm")
print(f"  Max daily rainfall after correction: {daily_stats['rainfall_daily_mm'].max():.1f} mm")

# 2. Fix Apparent Temperature: It should be calculated, not zero
print("\n" + "="*60)
print("CHECKING APPARENT TEMPERATURE")
print("="*60)

if 'apparent_temp_mean_c' in daily_stats.columns:
    # Check if all values are zero - convert to scalar boolean
    is_all_zero = (daily_stats['apparent_temp_mean_c'] == 0).all()
    # Convert to Python boolean if it's a pandas boolean
    if hasattr(is_all_zero, 'item'):
        is_all_zero = is_all_zero.item()
    
    if is_all_zero:
        print("⚠️ Apparent temperature is all zeros. Calculating from weather data...")
        
        # Apparent temperature formula (simplified):
        # AT = Ta + 0.33*e - 0.7*ws - 4.0
        # where e = vapor pressure (hPa), ws = wind speed (m/s)
        
        # Convert wind speed from km/h to m/s
        wind_speed_ms = daily_stats['wind_speed_mean_kmh'] / 3.6
        
        # Calculate apparent temperature
        daily_stats['apparent_temp_calculated'] = (
            daily_stats['temp_mean_c'] + 
            0.33 * daily_stats['vapor_pressure_mean_hpa'] - 
            0.7 * wind_speed_ms - 
            4.0
        ).round(1)
        
        print("✓ Apparent temperature calculated using standard formula")
        print(f"  Sample calculated values: {daily_stats['apparent_temp_calculated'].head().tolist()}")
        print(f"  Mean calculated apparent temp: {daily_stats['apparent_temp_calculated'].mean():.1f}°C")
    else:
        print("✓ Apparent temperature data appears valid")
        print(f"  Mean apparent temp: {daily_stats['apparent_temp_mean_c'].mean():.1f}°C")
else:
    print("⚠️ Apparent temperature column not found")

# 3. Display corrected statistics
print("\n" + "="*60)
print("CORRECTED STATISTICAL SUMMARY")
print("="*60)

print("\nRainfall (Corrected):")
print(f"  Total rainfall: {daily_stats['rainfall_daily_mm'].sum():.1f} mm")
print(f"  Rainy days: {daily_stats['rain_occurred'].sum()} out of {len(daily_stats)} days")
print(f"  Rain probability: {daily_stats['rain_occurred'].mean()*100:.1f}%")
print(f"  Max daily rainfall: {daily_stats['rainfall_daily_mm'].max():.1f} mm")

# Only calculate rainy days stats if there are rainy days
rainy_days = daily_stats[daily_stats['rain_occurred']==1]['rainfall_daily_mm']
if len(rainy_days) > 0:
    print(f"  Average rainfall on rainy days: {rainy_days.mean():.1f} mm")
    print(f"  Median rainfall on rainy days: {rainy_days.median():.1f} mm")
    print(f"  75th percentile: {rainy_days.quantile(0.75):.1f} mm")
    print(f"  95th percentile: {rainy_days.quantile(0.95):.1f} mm")

# Temperature summary
print("\nTemperature (°C):")
print(f"  Mean daily temp: {daily_stats['temp_mean_c'].mean():.1f}")
print(f"  Max temp recorded: {daily_stats['temp_max_c'].max():.1f}")
print(f"  Min temp recorded: {daily_stats['temp_min_c'].min():.1f}")

if 'apparent_temp_calculated' in daily_stats.columns:
    print("\nApparent Temperature (Calculated):")
    print(f"  Mean apparent temp: {daily_stats['apparent_temp_calculated'].mean():.1f}°C")
    print(f"  Range: {daily_stats['apparent_temp_calculated'].min():.1f} to {daily_stats['apparent_temp_calculated'].max():.1f}°C")

print("\nHumidity:")
print(f"  Mean humidity: {daily_stats['humidity_mean_pct'].mean():.1f}%")
print(f"  Max humidity: {daily_stats['humidity_max_pct'].max():.1f}%")

print("\nWind:")
print(f"  Mean wind speed: {daily_stats['wind_speed_mean_kmh'].mean():.1f} km/h")
if 'wind_gust_max_kmh' in daily_stats.columns:
    print(f"  Max wind gust: {daily_stats['wind_gust_max_kmh'].max():.1f} km/h")

print("\nPressure (hPa):")
print(f"  Mean pressure: {daily_stats['pressure_mean_hpa'].mean():.1f}")

# 4. Identify extreme rainfall events
print("\n" + "="*60)
print("EXTREME RAINFALL EVENTS (>50mm)")
print("="*60)
extreme_rain = daily_stats[daily_stats['rainfall_daily_mm'] > 50].sort_values('rainfall_daily_mm', ascending=False)
print(f"Found {len(extreme_rain)} days with >50mm rainfall")
if len(extreme_rain) > 0:
    print("\nTop 10 highest rainfall days:")
    display_cols = ['Date_only', 'rainfall_daily_mm', 'temp_mean_c', 'humidity_mean_pct']
    available_cols = [col for col in display_cols if col in extreme_rain.columns]
    print(extreme_rain[available_cols].head(10))
    
    # Check if these are realistic for Melbourne
    melbourne_record = 150  # Melbourne's record daily rainfall is around 150mm
    max_rain = extreme_rain['rainfall_daily_mm'].max()
    if max_rain > melbourne_record * 2:
        print(f"\n⚠️ WARNING: Extremely high rainfall values detected!")
        print(f"   Maximum: {max_rain:.1f} mm")
        print(f"   Melbourne's record is ~{melbourne_record} mm")
        print("   These may be data errors. Consider investigating these dates in raw data")
    elif max_rain > melbourne_record:
        print(f"\n⚠️ Note: Some rainfall values exceed Melbourne's record")
        print(f"   Maximum: {max_rain:.1f} mm (record is ~{melbourne_record} mm)")
        print("   These could be extreme events or data errors")
    else:
        print(f"\n✓ Rainfall values appear realistic (max within reasonable range)")

# 5. Add monthly and seasonal analysis
print("\n" + "="*60)
print("SEASONAL RAINFALL PATTERNS")
print("="*60)

# Add month and season
daily_stats['Month'] = daily_stats['Date_only'].dt.month

def get_season(month):
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

daily_stats['Season'] = daily_stats['Month'].apply(get_season)

# Seasonal statistics
seasonal_stats = daily_stats.groupby('Season').agg({
    'rainfall_daily_mm': ['mean', 'sum', 'count'],
    'rain_occurred': 'mean',
    'temp_mean_c': 'mean'
}).round(2)

seasonal_stats.columns = ['_'.join(col).strip() for col in seasonal_stats.columns.values]
print("\nSeasonal rainfall patterns:")
print(seasonal_stats)

# 6. Save corrected data
daily_stats.to_csv('melbourne_daily_weather_corrected.csv', index=False)
print("\n✓ Corrected daily data saved to 'melbourne_daily_weather_corrected.csv'")

# 7. Quick visualization of corrected rainfall
print("\n" + "="*60)
print("CREATING CORRECTED VISUALIZATION")
print("="*60)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Corrected daily rainfall over time
axes[0, 0].bar(daily_stats['Date_only'], daily_stats['rainfall_daily_mm'], 
               width=1, color='blue', alpha=0.5)
axes[0, 0].axhline(y=daily_stats['rainfall_daily_mm'].mean(), color='red', 
                   linestyle='--', label=f"Mean: {daily_stats['rainfall_daily_mm'].mean():.1f}mm")
axes[0, 0].set_title('Corrected Daily Rainfall', fontsize=12)
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Rainfall (mm)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Distribution of corrected rainfall (only rainy days)
if len(rainy_days) > 0:
    axes[0, 1].hist(rainy_days, bins=50, edgecolor='black', alpha=0.7, color='blue')
    axes[0, 1].axvline(rainy_days.mean(), color='red', linestyle='--', 
                       label=f"Mean: {rainy_days.mean():.1f}mm")
    axes[0, 1].axvline(rainy_days.median(), color='green', linestyle='--', 
                       label=f"Median: {rainy_days.median():.1f}mm")
    axes[0, 1].set_title('Distribution of Rainfall on Rainy Days', fontsize=12)
    axes[0, 1].set_xlabel('Rainfall (mm)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Monthly rainfall patterns
monthly_rain = daily_stats.groupby('Month')['rainfall_daily_mm'].mean()
axes[1, 0].bar(monthly_rain.index, monthly_rain.values, color='skyblue', edgecolor='black')
axes[1, 0].set_title('Average Monthly Rainfall (Corrected)', fontsize=12)
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Average Rainfall (mm)')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Rain probability by season
season_rain_prob = daily_stats.groupby('Season')['rain_occurred'].mean()
season_order = ['Summer', 'Autumn', 'Winter', 'Spring']
season_rain_prob = season_rain_prob.reindex(season_order)
axes[1, 1].bar(season_rain_prob.index, season_rain_prob.values, color='lightgreen', edgecolor='black')
axes[1, 1].set_title('Probability of Rain by Season', fontsize=12)
axes[1, 1].set_xlabel('Season')
axes[1, 1].set_ylabel('Probability')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Visualizations created with corrected data!")
print("\n" + "="*60)
print("DATA CORRECTION COMPLETE!")
print("="*60)

# Print key findings summary
print("\n📊 KEY FINDINGS SUMMARY:")
print(f"  • Melbourne gets rain on {daily_stats['rain_occurred'].mean()*100:.1f}% of days")
if len(rainy_days) > 0:
    print(f"  • Average rainfall on rainy days: {rainy_days.mean():.1f} mm")
    print(f"  • Typical rainfall (median) on rainy days: {rainy_days.median():.1f} mm")
print(f"  • Total rainfall over {len(daily_stats)} days: {daily_stats['rainfall_daily_mm'].sum():.1f} mm")
print(f"  • Wettest season: {seasonal_stats['rainfall_daily_mm_sum'].idxmax()} " +
      f"({seasonal_stats.loc[seasonal_stats['rainfall_daily_mm_sum'].idxmax(), 'rainfall_daily_mm_sum']:.1f} mm total)")
print(f"  • Driest season: {seasonal_stats['rainfall_daily_mm_sum'].idxmin()} " +
      f"({seasonal_stats.loc[seasonal_stats['rainfall_daily_mm_sum'].idxmin(), 'rainfall_daily_mm_sum']:.1f} mm total)")
print(f"  • Average temperature: {daily_stats['temp_mean_c'].mean():.1f}°C")


CORRECTING DATA ISSUES

Recalculating rainfall correctly...
✓ Rainfall corrected: Now using daily maximum (not sum)
  Total rainfall after correction: 9390.8 mm
  Max daily rainfall after correction: 54.6 mm

CHECKING APPARENT TEMPERATURE


ValueError: can only convert an array of size 1 to a Python scalar

In [26]:
# Display column names for reference
print("\n" + "="*60)
print("DAILY STATISTICS COLUMNS")
print("="*60)


DAILY STATISTICS COLUMNS


In [28]:
# Basic statistics summary
print("\n" + "="*60)
print("BASIC STATISTICAL SUMMARY")
print("="*60)


BASIC STATISTICAL SUMMARY


In [47]:
if 'temp_mean_c' in daily_stats.columns:
    print("\nTemperature (°C):")
    print(f"  Mean daily temp: {daily_stats['temp_mean_c'].mean():.1f}")
    print(f"  Max temp recorded: {daily_stats['temp_max_c'].max():.1f}")
    print(f"  Min temp recorded: {daily_stats['temp_min_c'].min():.1f}")

if 'rainfall_daily_mm' in daily_stats.columns:
    print("\nRainfall:")
    print(f"  Total rainfall: {daily_stats['rainfall_daily_mm'].sum():.1f} mm")
    print(f"  Rainy days: {daily_stats['rain_occurred'].sum()} out of {len(daily_stats)} days")
    print(f"  Rain probability: {daily_stats['rain_occurred'].mean()*100:.1f}%")
    print(f"  Max daily rainfall: {daily_stats['rainfall_daily_mm'].max():.1f} mm")
    rainy_days_data = daily_stats[daily_stats['rain_occurred']==1]['rainfall_daily_mm']
    if len(rainy_days_data) > 0:
        print(f"  Average rainfall on rainy days: {rainy_days_data.mean():.1f} mm")
else:
    print("\n⚠️ Rainfall data not available in daily statistics")

if 'humidity_mean_pct' in daily_stats.columns:
    print("\nHumidity:")
    print(f"  Mean humidity: {daily_stats['humidity_mean_pct'].mean():.1f}%")
    if 'humidity_max_pct' in daily_stats.columns:
        print(f"  Max humidity: {daily_stats['humidity_max_pct'].max():.1f}%")

if 'wind_speed_mean_kmh' in daily_stats.columns:
    print("\nWind:")
    print(f"  Mean wind speed: {daily_stats['wind_speed_mean_kmh'].mean():.1f} km/h")
    if 'wind_gust_max_kmh' in daily_stats.columns:
        print(f"  Max wind gust: {daily_stats['wind_gust_max_kmh'].max():.1f} km/h")

if 'pressure_mean_hpa' in daily_stats.columns:
    print("\nPressure (hPa):")
    print(f"  Mean pressure: {daily_stats['pressure_mean_hpa'].mean():.1f}")
    if 'pressure_range_hpa' in daily_stats.columns:
        print(f"  Avg pressure range: {daily_stats['pressure_range_hpa'].mean():.1f}")


Temperature (°C):
  Mean daily temp: 15.9
  Max temp recorded: 43.5
  Min temp recorded: 0.5

Rainfall:
  Total rainfall: 395759.6 mm
  Rainy days: 2097 out of 4078 days
  Rain probability: 51.4%
  Max daily rainfall: 3015.8 mm
  Average rainfall on rainy days: 188.7 mm

Humidity:
  Mean humidity: 66.5%
  Max humidity: 100.0%

Wind:
  Mean wind speed: 19.4 km/h

Pressure (hPa):
  Mean pressure: 1015.9
  Avg pressure range: 19.2


In [37]:
# Save to CSV
daily_stats.to_csv('melbourne_daily_weather.csv', index=False)
print("\n✓ Daily data saved to 'melbourne_daily_weather.csv'")

print("\n" + "="*60)
print("DATA PREPARATION COMPLETE!")
print("="*60)


✓ Daily data saved to 'melbourne_daily_weather.csv'

DATA PREPARATION COMPLETE!


### Observations (so far...)

### Melbourne Weather Patterns

#### Temperature

Mean: 15.9°C (self-note: seems about right for Melbourne)

Range: 0.5°C to 43.5°C (cold winters, hot summers)

Temperature range averages ~11-12°C daily variation

#### Rainfall (self-note: hopefully accurate once corrected):

Expected total: ~5,000-7,000 mm over 11 years (~854 mm/year)

Rain probability: ~35-40% of days (Melbourne average)

Typical rainy day: 2-10 mm, with occasional heavy falls

#### Humidity:

Mean: 66.5% (moderately humid)

Can reach 100% (fog/rain conditions)

#### Wind:

Mean speed: 19.4 km/h (breezy, hopefully typical for Melbourne)

Expect gusts up to 80-100 km/h during storms